Artigo principal: https://dl.acm.org/doi/10.1145/3698300.3698316#Bib0015 

Dataset das instancias (citado pelo paper): https://github.com/wrqccc/FJSP-DRL/tree/main/data/BenchData/Brandimarte

![alt text](./imgs/paper_table2.png)

## Leitura da Instância

### Formato do arquivo `.fjs` (Flexible Job Shop)

- **Linha 1**: `num_jobs  num_machines  avg_machines_per_op`
- **Linhas seguintes** (uma por job):
  - Primeiro número: `n_ops` (quantidade de operações do job)
  - Para cada operação: `n_alt` seguido de `n_alt` pares `(máquina  tempo_processamento)`

As máquinas são indexadas a partir de **1** no arquivo.

In [16]:
from dataclasses import dataclass, field
from typing import List, Dict, Tuple


@dataclass
class Operation:
    """Uma operação de um job, com múltiplas alternativas de máquina."""
    job_id: int
    op_index: int
    # Dict: machine_id (1-based) -> processing_time
    machines: Dict[int, int] = field(default_factory=dict)

    def __repr__(self):
        opts = ", ".join(f"M{m}:{t}" for m, t in sorted(self.machines.items()))
        return f"Op({self.job_id},{self.op_index})[{opts}]"


@dataclass
class Job:
    """Um job composto por uma sequência de operações."""
    job_id: int
    operations: List[Operation] = field(default_factory=list)

    def __repr__(self):
        return f"Job {self.job_id}: {self.operations}"


@dataclass
class FJSPInstance:
    """Instância completa do Flexible Job Shop Scheduling Problem."""
    num_jobs: int
    num_machines: int
    jobs: List[Job] = field(default_factory=list)


def parse_fjs(filepath: str) -> FJSPInstance:
    """Lê um arquivo .fjs e retorna uma FJSPInstance."""
    with open(filepath, 'r') as f:
        tokens = f.read().split()

    idx = 0

    # Cabeçalho
    num_jobs = int(tokens[idx]);     idx += 1
    num_machines = int(tokens[idx]); idx += 1
    _avg = int(tokens[idx]);         idx += 1  # avg machines per op (ignorado na leitura)

    instance = FJSPInstance(num_jobs=num_jobs, num_machines=num_machines)

    for j in range(num_jobs):
        n_ops = int(tokens[idx]); idx += 1
        job = Job(job_id=j + 1)

        for o in range(n_ops):
            n_alt = int(tokens[idx]); idx += 1
            op = Operation(job_id=j + 1, op_index=o + 1)

            for _ in range(n_alt):
                machine = int(tokens[idx]); idx += 1
                time    = int(tokens[idx]); idx += 1
                op.machines[machine] = time

            job.operations.append(op)

        instance.jobs.append(job)

    return instance


# Lê a primeira instância
instance_test = parse_fjs('./instancias/BrandimarteMk1.fjs')
print(f"Instância BrandimarteMk1")
print(f"  Jobs     : {instance_test.num_jobs}")
print(f"  Máquinas : {instance_test.num_machines}")
print()

Instância BrandimarteMk1
  Jobs     : 10
  Máquinas : 6



In [2]:
# Exibe os jobs e operações da instância
for job in instance_test.jobs:
    print(f"Job {job.job_id:>2}  ({len(job.operations)} operações):")
    for op in job.operations:
        opts = "  |  ".join(f"M{m} → t={t}" for m, t in sorted(op.machines.items()))
        print(f"    Op {op.op_index}: [{opts}]")
    print()

Job  1  (6 operações):
    Op 1: [M1 → t=5  |  M3 → t=4]
    Op 2: [M2 → t=1  |  M3 → t=5  |  M5 → t=3]
    Op 3: [M3 → t=4  |  M6 → t=2]
    Op 4: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]
    Op 5: [M3 → t=1]
    Op 6: [M3 → t=6  |  M4 → t=3  |  M6 → t=6]

Job  2  (5 operações):
    Op 1: [M2 → t=6]
    Op 2: [M3 → t=1]
    Op 3: [M1 → t=2]
    Op 4: [M2 → t=6  |  M4 → t=6]
    Op 5: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]

Job  3  (5 operações):
    Op 1: [M2 → t=6]
    Op 2: [M3 → t=4  |  M6 → t=2]
    Op 3: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]
    Op 4: [M2 → t=6  |  M3 → t=4  |  M6 → t=6]
    Op 5: [M1 → t=1  |  M5 → t=5]

Job  4  (5 operações):
    Op 1: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]
    Op 2: [M2 → t=6]
    Op 3: [M3 → t=1]
    Op 4: [M2 → t=1  |  M3 → t=5  |  M5 → t=3]
    Op 5: [M3 → t=4  |  M6 → t=2]

Job  5  (6 operações):
    Op 1: [M2 → t=1  |  M3 → t=5  |  M5 → t=3]
    Op 2: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]
    Op 3: [M2 → t=6]
    Op 4: [M1 → t=5  |  M3 → t=4]
    O

---
## Algoritmo Genético para o FJSP

### Visão geral

O **Flexible Job Shop Scheduling Problem (FJSP)** exige decidir:
1. **Qual máquina** processa cada operação (dentre as elegíveis).
2. **Em que ordem** as operações são processadas em cada máquina.

O objetivo é **minimizar o makespan** (Cmax = término da última operação).

### Representação do Cromossomo (dois vetores)

Cada indivíduo é um `dict` com **duas partes**:

```
cromossomo = {
  'ms': [...],   # Machine Selection
  'os': [...]    # Operation Sequence
}
```

| Parte | Comprimento | Significado |
|-------|-------------|-------------|
| **MS** (Machine Selection) | n_ops_total | `ms[i]` = índice (0-based) da máquina escolhida para a i-ésima operação na ordem canônica, dentro da lista ordenada de máquinas elegíveis para aquela operação |
| **OS** (Operation Sequence) | n_ops_total | Permutação de job_ids com repetição; cada job_id aparece *n_ops* vezes; a ordem define a sequência de processamento |

**Exemplo** — se a Op 1 do Job 1 pode ser feita em M1 (t=5) ou M3 (t=4):  
→ `ms[0] = 0` usa M1,  `ms[0] = 1` usa M3

### Decodificação → Makespan

Percorre o vetor **OS** em ordem. Para cada job_id encontrado:
- Pega a próxima operação pendente daquele job
- Consulta `ms` para saber qual máquina usa
- Agenda a operação no instante `max(máquina_livre, job_livre)`

### Operadores Genéticos

| Operador | Parte MS | Parte OS |
|----------|----------|----------|
| **Seleção** | Torneio (k=3) — evita problemas com fitness negativo e converge melhor que roleta para scheduling |
| **Cruzamento** | Dois pontos | POX (Precedence Operation Crossover) — preserva a ordem relativa das operações de cada job |
| **Mutação** | Por gene: com prob `pmut_ms`, sorteia nova máquina aleatória | Com prob `pmut_os`, realiza swap entre dois genes |

### Critérios de Parada

- Número máximo de gerações (`n_gen`)
- **Parada antecipada**: `patience` gerações consecutivas sem melhoria do melhor makespan

### Referência

O makespan ótimo conhecido para **BrandimarteMk1** é **40**.

In [3]:
import random
import time
from copy import deepcopy


# ─────────────────────────────────────────────────────────────────────
# PRÉ-PROCESSAMENTO DA INSTÂNCIA
# ─────────────────────────────────────────────────────────────────────

def get_op_info(inst):
    # Ordem canônica das operações: job1_op1, job1_op2, ..., jobN_opM
    all_ops   = []
    ms_max    = []   # índice máximo válido de máquina por operação
    for job in inst.jobs:
        for op in job.operations:
            all_ops.append((job.job_id, op.op_index))
            ms_max.append(len(op.machines) - 1)
    op_to_idx = {entry: i for i, entry in enumerate(all_ops)}
    return all_ops, op_to_idx, len(all_ops), ms_max


# ─────────────────────────────────────────────────────────────────────
# DECODIFICAÇÃO: cromossomo → makespan
# ─────────────────────────────────────────────────────────────────────

def decode_makespan(inst, chromosome, all_ops, op_to_idx):
    ms            = chromosome['ms']
    os_seq        = chromosome['os']
    machine_avail = [0] * (inst.num_machines + 1)   # 1-indexed
    job_avail     = [0] * (inst.num_jobs + 1)        # 1-indexed
    job_op_ptr    = [0] * (inst.num_jobs + 1)        # próxima op de cada job

    for job_id in os_seq:
        ptr             = job_op_ptr[job_id]
        job_op_ptr[job_id] += 1
        op              = inst.jobs[job_id - 1].operations[ptr]
        canon_idx       = op_to_idx[(job_id, op.op_index)]
        eligible        = sorted(op.machines.keys())
        machine_id      = eligible[ms[canon_idx]]
        proc_time       = op.machines[machine_id]
        start           = max(machine_avail[machine_id], job_avail[job_id])
        end             = start + proc_time
        machine_avail[machine_id] = end
        job_avail[job_id]         = end

    return max(machine_avail)


# ─────────────────────────────────────────────────────────────────────
# FITNESS: quanto menor o makespan, maior o fitness (valor positivo)
# fitness(c) = upper_bound - makespan(c)
# ─────────────────────────────────────────────────────────────────────

def make_fitness_fn(inst, all_ops, op_to_idx):
    ub = sum(
        max(op.machines.values())
        for job in inst.jobs
        for op in job.operations
    )
    def fitness(chrom):
        return ub - decode_makespan(inst, chrom, all_ops, op_to_idx)
    return fitness


# Pré-processa a instância global
all_ops_g, op_to_idx_g, n_ops_g, ms_max_g = get_op_info(instance_test)
ub_g = sum(max(op.machines.values()) for job in instance_test.jobs for op in job.operations)

print(f'Total de operações : {n_ops_g}')
print(f'Upper bound (Cmax) : {ub_g}')

Total de operações : 55
Upper bound (Cmax) : 254


In [4]:
# ─────────────────────────────────────────────────────────────────────
# INICIALIZAÇÃO DA POPULAÇÃO
# ─────────────────────────────────────────────────────────────────────

def init_population_fjsp(pop_size, inst, ms_max):
    # os_base: lista base com cada job_id repetido n_ops vezes
    os_base = []
    for job in inst.jobs:
        os_base.extend([job.job_id] * len(job.operations))

    population = []
    for _ in range(pop_size):
        # MS: para cada operação, escolhe índice de máquina aleatório
        ms     = [random.randint(0, ms_max[i]) for i in range(len(ms_max))]
        # OS: embaralha os_base aleatoriamente (garante precedência dentro de cada job)
        os_seq = os_base[:]
        random.shuffle(os_seq)
        population.append({'ms': ms, 'os': os_seq})
    return population

### Operadores Genéticos

#### Seleção por Torneio
Sorteia `k` indivíduos aleatoriamente e retorna o de maior fitness.  
Vantagens sobre a roleta: funciona bem para fitness com escala variável, converge de forma mais robusta.

#### Cruzamento
- **MS — dois pontos**: sorteia dois pontos de corte `c1 < c2`; o filho recebe `ms1[:c1] + ms2[c1:c2] + ms1[c2:]`
- **OS — POX** (Precedence Operation Crossover): sorteia um subconjunto `S` de jobs; o filho herda as posições de `S` diretamente do parent1 (mesma ordem relativa) e preenche as demais com os jobs ∉ S na ordem em que aparecem no parent2. Isso **garante que a sequência de operações de cada job seja válida**.

#### Mutação
- **MS**: para cada gene, com probabilidade `pmut_ms`, sorteia um novo índice de máquina aleatório
- **OS**: com probabilidade `pmut_os`, troca dois genes aleatórios (swap)

In [5]:
# ─────────────────────────────────────────────────────────────────────
# SELEÇÃO POR TORNEIO
# ─────────────────────────────────────────────────────────────────────

def tournament_select(population, fitness_fn, k=3):
    candidates = random.sample(population, k)
    return max(candidates, key=fitness_fn)


# ─────────────────────────────────────────────────────────────────────
# CRUZAMENTO
# ─────────────────────────────────────────────────────────────────────

def crossover_ms(ms1, ms2):
    # Cruzamento de dois pontos para a parte MS
    n      = len(ms1)
    c1, c2 = sorted(random.sample(range(n + 1), 2))
    return ms1[:c1] + ms2[c1:c2] + ms1[c2:]


def pox_crossover_os(os1, os2, num_jobs):
    # POX: sorteia subset S de jobs;
    #   filho herda posicoes de S do parent1 (preserva ordem relativa de S),
    #   restantes preenchidas com jobs nao em S na ordem do parent2.
    k       = random.randint(1, num_jobs - 1)
    job_set = set(random.sample(range(1, num_jobs + 1), k))
    p2_iter = (j for j in os2 if j not in job_set)
    child   = []
    for j in os1:
        child.append(j if j in job_set else next(p2_iter))
    return child


def crossover_fjsp(c1, c2, num_jobs, prob=0.8):
    # Aplica cruzamento com probabilidade prob; retorna dois filhos
    if random.random() > prob:
        return deepcopy(c1), deepcopy(c2)
    ms1 = crossover_ms(c1['ms'], c2['ms'])
    ms2 = crossover_ms(c2['ms'], c1['ms'])
    os1 = pox_crossover_os(c1['os'], c2['os'], num_jobs)
    os2 = pox_crossover_os(c2['os'], c1['os'], num_jobs)
    return {'ms': ms1, 'os': os1}, {'ms': ms2, 'os': os2}


# ─────────────────────────────────────────────────────────────────────
# MUTAÇÃO
# ─────────────────────────────────────────────────────────────────────

def mutate_ms(ms, ms_max, pmut):
    # Por gene: com prob pmut, reatribui maquina aleatoria
    new_ms = ms[:]
    for i in range(len(new_ms)):
        if ms_max[i] > 0 and random.random() < pmut:
            new_ms[i] = random.randint(0, ms_max[i])
    return new_ms


def mutate_os(os_seq, pmut):
    # Com prob pmut, troca dois genes aleatorios (swap)
    if random.random() >= pmut:
        return os_seq[:]
    new_os = os_seq[:]
    i, j   = random.sample(range(len(new_os)), 2)
    new_os[i], new_os[j] = new_os[j], new_os[i]
    return new_os


def mutate_fjsp(chrom, ms_max, pmut_ms=0.05, pmut_os=0.1):
    return {
        'ms': mutate_ms(chrom['ms'], ms_max, pmut_ms),
        'os': mutate_os(chrom['os'], pmut_os),
    }

### Loop Principal do AG

Estrutura a cada geração:
1. **Elitismo**: o melhor indivíduo da geração anterior é copiado diretamente (preserva o melhor)
2. **Seleção → Cruzamento → Mutação**: pares de pais geram dois filhos até completar `pop_size`
3. Atualiza o melhor global e incrementa o contador de gerações sem melhoria
4. Critério de parada antecipada se `no_improve >= patience`

In [6]:
def genetic_algorithm_fjsp(
    inst,
    pop_size       = 100,
    n_gen          = 300,
    crossover_prob = 0.8,
    pmut_ms        = 0.05,
    pmut_os        = 0.1,
    tournament_k   = 3,
    elitism        = True,
    patience       = 80,
    seed           = 42,
    verbose        = True,
):
    random.seed(seed)

    # pré-processamento
    all_ops, op_to_idx, _, ms_max = get_op_info(inst)
    fitness_fn = make_fitness_fn(inst, all_ops, op_to_idx)

    # inicialização
    population    = init_population_fjsp(pop_size, inst, ms_max)
    best_chrom    = max(population, key=fitness_fn)
    best_makespan = decode_makespan(inst, best_chrom, all_ops, op_to_idx)
    history       = [best_makespan]
    no_improve    = 0

    for gen in range(n_gen):
        new_pop = []

        # elitismo: preserva o melhor indivíduo
        if elitism:
            new_pop.append(deepcopy(best_chrom))

        # gera novos individuos: selecao → cruzamento → mutacao
        while len(new_pop) < pop_size:
            p1 = tournament_select(population, fitness_fn, tournament_k)
            p2 = tournament_select(population, fitness_fn, tournament_k)
            child1, child2 = crossover_fjsp(p1, p2, inst.num_jobs, crossover_prob)
            child1 = mutate_fjsp(child1, ms_max, pmut_ms, pmut_os)
            child2 = mutate_fjsp(child2, ms_max, pmut_ms, pmut_os)
            new_pop.append(child1)
            if len(new_pop) < pop_size:
                new_pop.append(child2)

        population = new_pop

        # atualiza melhor global
        gen_best      = max(population, key=fitness_fn)
        gen_makespan  = decode_makespan(inst, gen_best, all_ops, op_to_idx)

        if gen_makespan < best_makespan:
            best_makespan = gen_makespan
            best_chrom    = deepcopy(gen_best)
            no_improve    = 0
        else:
            no_improve += 1

        history.append(best_makespan)

        if verbose and (gen + 1) % 50 == 0:
            print(f'  Geracao {gen + 1:>4} | Melhor makespan: {best_makespan}')

        # criterio de parada antecipada
        if patience and no_improve >= patience:
            if verbose:
                print(f'  Parada antecipada na geracao {gen + 1} '
                      f'(sem melhoria por {patience} geracoes)')
            break

    return best_chrom, best_makespan, history, all_ops, op_to_idx

In [22]:
PARAMS = dict(
    inst           = parse_fjs('./instancias/BrandimarteMk3.fjs'),
    pop_size       = 1000,
    n_gen          = 1000,
    crossover_prob = 0.5,
    pmut_ms        = 0.05,
    pmut_os        = 0.2,
    tournament_k   = 4,
    elitism        = True,
    patience       = 300,
    seed           = 42,
)

print('Executando AG — BrandimarteMk1')
print(f'  {PARAMS["inst"].num_jobs} jobs | {PARAMS["inst"].num_machines} maquinas | {n_ops_g} operacoes')
print()

t0 = time.time()
best_chrom, best_mk, history, all_ops, op_to_idx = genetic_algorithm_fjsp(**PARAMS)
elapsed = time.time() - t0

print()
print(f'Makespan encontrado           : {best_mk}')
print(f'Referencia (otimo conhecido)  : 40')
print(f'Gap em relacao ao otimo       : {best_mk - 40}')
print(f'Tempo de execucao             : {elapsed:.2f}s')

Executando AG — BrandimarteMk1
  15 jobs | 8 maquinas | 55 operacoes

  Geracao   50 | Melhor makespan: 232
  Geracao  100 | Melhor makespan: 209
  Geracao  150 | Melhor makespan: 204
  Geracao  200 | Melhor makespan: 204
  Geracao  250 | Melhor makespan: 204
  Geracao  300 | Melhor makespan: 204
  Geracao  350 | Melhor makespan: 204
  Geracao  400 | Melhor makespan: 204
  Parada antecipada na geracao 418 (sem melhoria por 300 geracoes)

Makespan encontrado           : 204
Referencia (otimo conhecido)  : 40
Gap em relacao ao otimo       : 164
Tempo de execucao             : 82.31s


In [23]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(history))),
    y=history,
    mode='lines',
    name='Melhor makespan por geracao',
    line=dict(color='steelblue', width=2)
))
fig.add_hline(
    y=40, line_dash='dash', line_color='red',
    annotation_text='Otimo conhecido (Cmax = 40)',
    annotation_position='top right'
)
fig.update_layout(
    title='Convergencia do AG — BrandimarteMk1',
    xaxis_title='Geracao',
    yaxis_title='Makespan (Cmax)',
    height=400,
    template='plotly_white',
)
fig.show()

In [24]:
import pandas as pd


def decode_schedule_detail(inst, chromosome, all_ops, op_to_idx):
    # Retorna lista de (job_id, op_index, machine_id, start, end)
    ms            = chromosome['ms']
    os_seq        = chromosome['os']
    machine_avail = [0] * (inst.num_machines + 1)
    job_avail     = [0] * (inst.num_jobs + 1)
    job_op_ptr    = [0] * (inst.num_jobs + 1)
    schedule      = []
    for job_id in os_seq:
        ptr             = job_op_ptr[job_id]
        job_op_ptr[job_id] += 1
        op              = inst.jobs[job_id - 1].operations[ptr]
        canon_idx       = op_to_idx[(job_id, op.op_index)]
        eligible        = sorted(op.machines.keys())
        machine_id      = eligible[ms[canon_idx]]
        proc_time       = op.machines[machine_id]
        start           = max(machine_avail[machine_id], job_avail[job_id])
        end             = start + proc_time
        machine_avail[machine_id] = end
        job_avail[job_id]         = end
        schedule.append((job_id, op.op_index, machine_id, start, end))
    return schedule


schedule = decode_schedule_detail(PARAMS["inst"], best_chrom, all_ops, op_to_idx)
df_sched = pd.DataFrame(schedule, columns=['Job', 'Operacao', 'Maquina', 'Inicio', 'Fim'])
display(df_sched.sort_values(['Maquina', 'Inicio']).reset_index(drop=True))
print(f'\nMakespan (Cmax) = {df_sched["Fim"].max()}')

,Job,Operacao,Maquina,Inicio,Fim
0,10,1,1,0,17
1,9,1,1,17,34
2,2,2,1,34,51
3,12,4,1,51,68
4,8,2,1,68,85
...,...,...,...,...,...
145,6,8,8,144,155
146,2,9,8,155,158
147,1,9,8,170,175
148,8,6,8,175,178



Makespan (Cmax) = 204


In [25]:
import plotly.express as px


def plot_gantt_plotly(schedule, num_jobs, num_machines, makespan):
    palette = px.colors.qualitative.Plotly
    machine_order = [f'M{m}' for m in range(num_machines, 0, -1)]  # M6..M1 de cima p/ baixo

    fig = go.Figure()
    seen_jobs = set()

    for job_id, op_idx, machine_id, start, end in schedule:
        color        = palette[(job_id - 1) % len(palette)]
        show_legend  = job_id not in seen_jobs
        seen_jobs.add(job_id)

        fig.add_trace(go.Bar(
            orientation='h',
            y=[f'M{machine_id}'],
            x=[end - start],
            base=[start],
            name=f'Job {job_id}',
            legendgroup=f'Job {job_id}',
            showlegend=show_legend,
            marker=dict(color=color, line=dict(color='black', width=0.5)),
            text=f'J{job_id}O{op_idx}',
            textposition='inside',
            insidetextanchor='middle',
            textfont=dict(size=8, color='white'),
            hovertemplate=(
                f'<b>Job {job_id} — Op {op_idx}</b><br>'
                f'Maquina : M{machine_id}<br>'
                f'Inicio  : {start}<br>'
                f'Fim     : {end}<br>'
                f'Duracao : {end - start}<extra></extra>'
            ),
        ))

    fig.update_layout(
        barmode='overlay',
        title=f'Diagrama de Gantt — BrandimarteMk1  (Cmax = {makespan})',
        xaxis=dict(title='Tempo', range=[0, makespan * 1.06]),
        yaxis=dict(
            title='Maquina',
            categoryorder='array',
            categoryarray=machine_order,
        ),
        legend_title='Job',
        height=450,
        template='plotly_white',
    )
    fig.add_vline(
        x=makespan, line_dash='dash', line_color='red',
        annotation_text=f'Cmax={makespan}', annotation_position='top right'
    )
    fig.show()


plot_gantt_plotly(schedule, PARAMS["inst"].num_jobs, PARAMS["inst"].num_machines, best_mk)